# Text Emotion Classifier - DistilBERT (GoEmotions to 6 classes)
## Google Colab Version

Fine-tunes `distilbert-base-uncased` on GoEmotions with a 28 to 6 label mapping.

**Classes:** Positive, Neutral, Stress, Anxiety, Negative, Depression

---
### Setup Instructions
1. Go to **Runtime > Change runtime type** and select **T4 GPU**
2. Run all cells in order (**Runtime > Run all**)
3. When prompted, authorize Google Drive access
4. Model saves to `MyDrive/BE_models/text_final_6/` for persistence
---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory in Drive
import os
DRIVE_PATH = '/content/drive/MyDrive/BE_models/text_final_6'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Model will be saved to: {DRIVE_PATH}')

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn matplotlib seaborn

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow!')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ds = load_dataset("google-research-datasets/go_emotions")
train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# 28-class GoEmotions labels (id → name)
id2label_28 = {
    0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval',
    5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment',
    10: 'disapproval', 11: 'disgust', 12: 'embarrassment', 13: 'excitement',
    14: 'fear', 15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness',
    20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse',
    25: 'sadness', 26: 'surprise', 27: 'neutral'
}

# Map 28 classes → 6 classes
final_map = {
    'admiration':0, 'amusement':0, 'approval':0, 'caring':0, 'desire':0,
    'excitement':0, 'gratitude':0, 'joy':0, 'love':0, 'optimism':0,
    'pride':0, 'relief':0,               # → Positive (0)
    'curiosity':1, 'realization':1, 'surprise':1, 'neutral':1,  # → Neutral (1)
    'anger':2, 'annoyance':2, 'disapproval':2, 'confusion':2,   # → Stress (2)
    'fear':3, 'nervousness':3,            # → Anxiety (3)
    'disappointment':4, 'disgust':4, 'embarrassment':4, 'remorse':4,  # → Negative (4)
    'grief':5, 'sadness':5                # → Depression (5)
}

class_names = ['Positive', 'Neutral', 'Stress', 'Anxiety', 'Negative', 'Depression']

def convert_label(example):
    labels = example["labels"]
    if len(labels) == 0:
        example["label"] = 1  # default to Neutral
    else:
        first = labels[0]
        emotion = id2label_28[first]
        example["label"] = final_map[emotion]
    return example

train_ds = train_ds.map(convert_label)
val_ds = val_ds.map(convert_label)
test_ds = test_ds.map(convert_label)
print("Label mapping complete")

In [ ]:
from collections import Counter
train_labels = [ex['label'] for ex in train_ds]
counts = Counter(train_labels)
print("Class distribution (train):")
for i in range(6):
    print(f"  {class_names[i]:12s}: {counts[i]:6d}")

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
keep = ["input_ids", "attention_mask", "label"]

train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Features: {train_ds.column_names}")

In [ ]:
# Load model with increased dropout to combat overfitting
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name, num_labels=6)
config.dropout = 0.2          # increase from default 0.1
config.attention_dropout = 0.2
config.seq_classif_dropout = 0.3  # classifier-specific dropout

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config
)
print(f"Model loaded: {model_name}")
print(f"Dropout: {config.dropout}, Attention dropout: {config.attention_dropout}")

In [ ]:
acc = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

In [ ]:
import torch
has_cuda = torch.cuda.is_available()
print(f"CUDA available: {has_cuda}")

args = TrainingArguments(
    output_dir=DRIVE_PATH,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,

    weight_decay=0.1,              # increased from 0.01 to combat overfitting
    warmup_ratio=0.1,
    label_smoothing_factor=0.05,   # add label smoothing

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,
    logging_dir=DRIVE_PATH + "/logs",

    fp16=has_cuda,                 # only use if GPU available
    dataloader_pin_memory=False,   # avoid warnings on CPU
    report_to="none",
)
print(f"Weight decay: {args.weight_decay}")
print(f"Label smoothing: {args.label_smoothing_factor}")

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
print("Trainer initialized")

In [ ]:
trainer.train()

In [ ]:
# Evaluate the best model on the test set
test_results = trainer.evaluate(test_ds)
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test F1:       {test_results['eval_f1']:.4f}")

In [ ]:
# Get detailed predictions for the test set
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(f"Predictions shape: {y_pred.shape}")
print(f"True labels shape: {y_true.shape}")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\n{'='*50}")
print(f"Test Set Performance")
print(f"{'='*50}")
print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")

print(f"\n{'='*50}")
print("Classification Report")
print(f"{'='*50}")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Text Model (GoEmotions → 6 classes)")
plt.tight_layout()
plt.savefig(DRIVE_PATH + "/text_confusion_matrix.png", dpi=150)
plt.show()
print(f"Confusion matrix saved to {DRIVE_PATH}/text_confusion_matrix.png")

In [ ]:
model.save_pretrained(DRIVE_PATH)
tokenizer.save_pretrained(DRIVE_PATH)
print(f"Model and tokenizer saved to {DRIVE_PATH}/")

In [ ]:
# Quick sanity check: load saved model and run inference
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=DRIVE_PATH,
    tokenizer=DRIVE_PATH,
    return_all_scores=True
)

test_texts = [
    "I'm so happy today!",
    "This is terrible, I'm really angry",
    "I feel anxious about the exam",
    "I'm feeling quite neutral about it",
    "I'm so sad and lonely"
]

print("\n" + "="*60)
print("Sanity Check — Inference on sample texts")
print("="*60)
for text in test_texts:
    result = classifier(text)[0]
    top = max(result, key=lambda x: x['score'])
    class_idx = int(top['label'].split('_')[1])
    print(f"\nText: {text}")
    print(f"  Predicted: {class_names[class_idx]:12s} ({top['score']:.4f})")
    # Show top 3
    sorted_results = sorted(result, key=lambda x: x['score'], reverse=True)[:3]
    for r in sorted_results:
        idx = int(r['label'].split('_')[1])
        print(f"    {class_names[idx]:12s}: {r['score']:.4f}")

---
### Download model back to local project

**Option 1: Colab File Browser**
1. Click the **Files** icon in the left sidebar
2. Navigate to `drive/MyDrive/BE_models/text_final_6/`
3. Right-click the folder → **Download**
4. Unzip and place in your local `backend/models/text_final_6/`

**Option 2: Programmatic download** (run this cell locally after training):
```python
from google.colab import files
import shutil
# Zip the model folder first
!zip -r /tmp/text_final_6.zip /content/drive/MyDrive/BE_models/text_final_6/
files.download('/tmp/text_final_6.zip')
```
---